## Demo Notebook

In [1]:
import pandas as pd

from utils.download import *
from pulearn.pubase import *


download_from_gdrive("deceptive-opinion.csv")

documents = pd.read_csv("../data/deceptive-opinion.csv")
documents.head()

[~_o] File already exists --> Aborting download


,deceptive,hotel,polarity,source,text
0,truthful,conrad,positive,TripAdvisor,We stayed for a one night getaway with family ...
1,truthful,hyatt,positive,TripAdvisor,Triple A rate with upgrade to view room was le...
2,truthful,hyatt,positive,TripAdvisor,This comes a little late as I'm finally catchi...
3,truthful,omni,positive,TripAdvisor,The Omni Chicago really delivers on all fronts...
4,truthful,hyatt,positive,TripAdvisor,I asked for a high floor away from the elevato...


Drop not needed columns

In [2]:
documents = documents.drop(columns=["hotel", "source", "polarity"])
documents["deceptive"] = documents["deceptive"].apply(lambda x: 0 if x == "truthful" else 1)
documents.head()

,deceptive,text
0,0,We stayed for a one night getaway with family ...
1,0,Triple A rate with upgrade to view room was le...
2,0,This comes a little late as I'm finally catchi...
3,0,The Omni Chicago really delivers on all fronts...
4,0,I asked for a high floor away from the elevato...


Isolate all deceptive documents and extract a partial test set of 160 instances

In [3]:
deceptive_docs = documents.loc[(documents["deceptive"] == 1)]
deceptive_docs.reset_index(drop=True, inplace=True)
indices = extract_sample(deceptive_docs["deceptive"], ratio=0.2, value=1)
deceptive_set = deceptive_docs.loc[indices]
deceptive_set

,deceptive,text
213,1,My stay at this hotel was one of the best I ha...
309,1,The Hilton Chicago is one of the best Hotels I...
270,1,I am often traveling on business and I always ...
127,1,My family really enjoyed this hotel the weeken...
193,1,I loved staying at the Hard Rock Hotel in Chic...
...,...,...
620,1,I booked my weekend Chicago stay at the Hard R...
71,1,My husband and I stayed here for a short get-a...
604,1,My experience at the Ambassador East Hotel was...
404,1,"The room was very pretty, and the location is ..."


Isolate all non-deceptive documents and extract a partial test set of another 160 instances

In [4]:
non_deceptive_docs = documents.loc[(documents["deceptive"] == 0)]
non_deceptive_docs.reset_index(drop=True, inplace=True)
indices = extract_sample(non_deceptive_docs["deceptive"], ratio=0.2, value=0)
non_deceptive_set = non_deceptive_docs.loc[indices]
non_deceptive_set

,deceptive,text
80,0,The sheraton was a wonderful hotel! When me an...
359,0,My wife and I stayed for 2 nights and loved it...
412,0,"The location of the hotel is great, right on t..."
257,0,We just got back from 3 nights at the Sofitel....
190,0,We also just returned from a frigid weekend in...
...,...,...
151,0,I stayed at the Swissotel for a business trip ...
55,0,We stayed at the Omni between Christmas and Ne...
578,0,"Apart from the location, great, but it stops t..."
291,0,"Hotel Monaco is in a really great location, on..."


Merge above sets to create a unified test set

In [5]:
test_set = pd.concat([deceptive_set, non_deceptive_set], ignore_index=True)
test_set

,deceptive,text
0,1,My stay at this hotel was one of the best I ha...
1,1,The Hilton Chicago is one of the best Hotels I...
2,1,I am often traveling on business and I always ...
3,1,My family really enjoyed this hotel the weeken...
4,1,I loved staying at the Hard Rock Hotel in Chic...
...,...,...
315,0,I stayed at the Swissotel for a business trip ...
316,0,We stayed at the Omni between Christmas and Ne...
317,0,"Apart from the location, great, but it stops t..."
318,0,"Hotel Monaco is in a really great location, on..."


Accordingly extract a training set and turn it into a PU dataset

In [6]:
train_set = documents.loc[~(documents["text"].isin(test_set["text"]))]
train_set = train_set.reset_index(drop=True)

indices = extract_sample(train_set["deceptive"], ratio=0.37, value=1)
train_set["pu-label"] = 0
train_set.loc[indices, "pu-label"] = 1

train_set["pu-label"].value_counts()

0    1041
1     237
Name: pu-label, dtype: int64

Define a text preprocessing function

In [7]:
import re

from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

def prepare(row):
    filtered = []
    row = row.lower().strip()
    row = re.sub('[^A-Za-z0-9.]+', ' ', row)
    row_parts = row.split()
    for part in row_parts:
        filtered.append(lemmatizer.lemmatize(part))
    return " ".join(filtered)

Create TF-IDF vectors

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

X_train = train_set["text"]
X_train = X_train.apply(lambda row: prepare(row))
y_train = train_set["pu-label"]

X_test = test_set["text"]
X_test = X_test.apply(lambda row: prepare(row))
y_test = test_set["deceptive"]

tfidf_input = pd.concat([X_train, X_test])

vectorizer = TfidfVectorizer()

vectors = vectorizer.fit_transform(tfidf_input)
feature_names = vectorizer.get_feature_names_out()
dense = vectors.todense().tolist()

X = pd.DataFrame(dense, columns=feature_names)

train_rows = X_train.shape[0]
test_rows = X_test.shape[0]

X_train = X.iloc[:train_rows]
X_test = X.iloc[-test_rows:]
X_test = X_test.reset_index(drop=True)

Growing classifier

In [9]:
from pulearn.iterative_classifiers import *
from sklearn.metrics import f1_score


roc_svm = GrowingClassifier("Rocchio", "SVC", params1={"metric": "manhattan"}, params2={"kernel": "linear"})
roc_svm.fit(X_train, y_train)
predictions = roc_svm.predict(X_test)

print("\nF1-score = {:.2f}".format(f1_score(y_test, predictions, average="macro") * 100))

Iteration: 0
[[  0 250]
 [  1 149]]
Iteration: 1
[[ 0 81]
 [ 1 68]]
Iteration: 2
[[ 0 46]
 [ 1 22]]
Iteration: 3
[[ 0 16]
 [ 1  6]]
Iteration: 4
[[0 5]
 [1 1]]
Iteration: 5
[[1 1]]
[[  0 256]
 [  1  64]]

F1-score = 65.66


Pruning Classifier -- currently contains a bug which results in an endless loop

Reason: internal 2nd estimator **sometimes** makes only negative predictions & results in while condition never exiting

In [10]:
roc_svm = PruningClassifier("Rocchio", "SVC", params1={"metric": "manhattan"}, params2={"kernel": "linear"})
roc_svm.fit(X_train, y_train)
predictions = roc_svm.predict(X_test)

print("\nF1-score = {:.2f}".format(f1_score(y_test, predictions, average="macro") * 100))

Iteration: 0
[[  0 641]
 [  1   1]]
[[  0 256]
 [  1  64]]

F1-score = 65.66
